**TOPOGRAFI**

In [ ]:
'''import library yang diperlukan'''

import ee        # untuk mengakses dataset citra satelit dari GEE
import geemap    # untuk keperluan visualisasi
import folium    # menampilkan peta interaktif

In [ ]:
# otentikasi ke akun GEE

ee.Authenticate()
ee.Initialize(project='praktikum6ask-475512')  # bagian ini sesuaikan dengan nama akun GEE

1. Akses Data

In [ ]:
# Mendefinisikan batas wilayah Indonesia
koordinat = [[
    [141, -11], [141, 6], [95, 6], [95, -11]
]]
fc = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Polygon(koordinat),
        {'name': 'Indonesia', 'fill': 1}
    )
])

image = ee.Image().paint(fc, 'fill').paint(fc, 3, 5).toByte()

# Mengambil batas administrasi Indonesia dari dataset FAO GAUL
country = ee.FeatureCollection("FAO/GAUL/2015/level0")
roi = country.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')).geometry()

# Menentukan rentang waktu
Tahunmulai = 2022
Tahunakhir = 2023

tanggalmulai = ee.Date.fromYMD(Tahunmulai, 1, 1)
tanggalakhir = ee.Date.fromYMD(Tahunakhir + 1, 1, 1)

tahun = ee.List.sequence(Tahunmulai, Tahunakhir)
bulan = ee.List.sequence(1, 12)

# Akses Global DEM 30 m Copernicus
DEM = ee.ImageCollection("COPERNICUS/DEM/GLO30") \
          .mosaic() \
          .select('DEM') \
          .clip(roi)

2. Peta Ketinggian Indonesia

In [ ]:
# Parameter visualisasi DEM
dem_vis = {
    'min': 0,
    'max': 5000,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red', 'brown']
}

# Peta Interaktif
Map = geemap.Map(center=[-2, 118], zoom=5)
Map.addLayer(DEM, dem_vis, "Elevation (DEM)")

# Menambahkan legenda
Map.add_colorbar(vis_params=dem_vis, label="Elevation (m)", orientation='horizontal')
Map.addLayerControl()

# Menampilkan peta
Map

Map(center=[-2, 118], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

3. Memilih WIlayah Kajian

In [ ]:
# Akses SHP yang ada di GEE
shp = ee.FeatureCollection("projects/praktikum6ask-475512/assets/shpkabupaten")
region = shp.filter(ee.Filter.eq('KABUPATEN', 'LUMAJANG')).geometry();

# Akses data DEM Copernicus GLO-30 clip dengan region
DEM = ee.ImageCollection("COPERNICUS/DEM/GLO30") \
          .mosaic() \
          .select('DEM') \
          .clip(region)   # clip ke region

# Parameter visualisasi
dem_vis = {
    'min': 0,
    'max': 4000,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red', 'brown']
}

# Buat peta
Map = geemap.Map(center=[-8.13076628230708, 113.21677352220001], zoom=9)  # perkiraan pusat Lumajang, zoom bisa diubah ubah
Map.addLayer(region, {'color': 'black'}, "Boundary - Lumajang")
Map.addLayer(DEM, dem_vis, "Elevation Lumajang (DEM)")

# Tambahkan layer batas wilayah
Map.add_colorbar(vis_params=dem_vis, label="Elevation (m)", orientation='horizontal')
Map.addLayerControl()

Map

Map(center=[-8.13076628230708, 113.21677352220001], controls=(WidgetControl(options=['position', 'transparent_…

4. Slope

In [ ]:
# Mengambil data elevasi global SRTM (Shuttle Radar Topography Mission) dengan resolusi 30 meter.
region = region.buffer(10)
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')

# Hitung slope (kemiringan)
slope = ee.Terrain.slope(dem)

# Clip slope ke wilayah kajian
slope_clip = slope.clip(region)

# Cek nilai maksimum dan minimum slope
stat = slope_clip.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=region,
    scale=30,
    maxPixels=1e9
)

print("Range slope:", stat.getInfo())

Range slope: {'slope_max': 74.58668518066406, 'slope_min': 0}


In [ ]:
import folium    # menampilkan peta interaktif
from geemap import foliumap as geemap

slope_vis = {
    'min': 0,
    'max': 20, # pastikan nilai max nya melingkupi slope_max yang sudah dicek
    'palette': ['blue', 'green', 'yellow', 'orange', 'red', 'brown']
}

# Buat peta
Map_Slope = geemap.Map(center=[-8.13076628230708, 113.21677352220001], zoom=9)  # perkiraan pusat Lumajang, zoom bisa diubah ubah
Map_Slope.addLayer(region, {'color': 'black'}, "Boundary - Lumajang")
Map_Slope.addLayer(slope_clip, slope_vis, "Slope Lumajang")

# Tambahkan layer batas wilayah
Map_Slope.add_colorbar(vis_params=slope_vis, label="Slope (°)", orientation='horizontal')
Map_Slope.addLayerControl()

Map_Slope

5. TPI (Topographic Position Index)

In [ ]:
region_simplified = region.simplify(100).buffer(1000)

# Gunakan data DEM Copernicus GLO30 dan clip ke Cianjur lagi
DEM = ee.ImageCollection("COPERNICUS/DEM/GLO30") \
    .mosaic() \
    .select('DEM') \
    .clip(region_simplified)

# Fungsi TPI berbasis kernel piksel
def calculate_tpi(image, radius=3):
    kernel = ee.Kernel.square(radius, units='pixels', normalize=True)
    mean_elev = image.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=kernel
    )
    tpi = image.subtract(mean_elev)
    return tpi

# Kalkulasi TPI dan clip
tpi = calculate_tpi(DEM, radius=3).clip(region_simplified)

# Parameter visualisasi
tpi_vis = {
    'min': -50,
    'max': 50,
    'palette': ['blue', 'yellow', 'red']  # biru = lembah, kuning = datar, merah = punggungan
}

# Buat peta
Map = geemap.Map(center=[-8.13076628230708, 113.21677352220001], zoom=9)
Map.addLayer(region, {'color': 'black'}, 'Boundary - Lumajang')
Map.addLayer(tpi, tpi_vis, 'Topographic Position Index (TPI)')

# Tambahkan legenda
Map.add_legend(
    title='Topographic Position Index (TPI)',
    colors=['#0000FF', '#FFFF00', '#FF0000'],  # biru, kuning, merah
    labels=['Lembah', 'Dataran', 'Punggungan']
)

Map.addLayerControl()
Map
